<center>
    <h2>Evaluation protocole for the 2026 IA Pau Data Battle</h2>
</center>

---

This notebook provides you a protocol to evaluate your model for the data battle


### Why this notebook

This is not:

- The only and perfect way to evaluate your model.

The subject of evaluation is not trivial and there are different methods with their pro and con. The construction of the relevant method for evaluation is <span style="color:red">part of the subject of the data battle</span>. For instance, proposing a new relevant version of the risk estimation compatible with AI models would be very interesting.

- A mandatory step to participate to the Jury. You do not have to use this notebook to evaluate your models (but you may want). 

This is for :

- To provide a common evaluation framework. This enables the jury to better understand your results on the metric they understand and will see in other groups.

- Provide trust : we will send you blind test data which consists of subset of alerts. We will use the procedure described in this notebook to evaluate your results based on the prediction you will send to us. This enables us to validate your prediction.
 

### Two criteria : Risk and time gain 

As you know, our data from Meteorage consists of the location and date of lightning strikes from thunderstorms passing around areas to be protected (airports).

We assume :

- The last lightning of the alert $a$ is at time $t^a$ (all times and dates are in minutes)

- That a model produces $N^a$ predictions $p^a_i, i=1..N^a$ for the end of alert $a$. Each prediction $p^a_i$ is composed of a confidence score $s^a_i$, the date at which the prediction is emitted $d_i^a$ and the value of the prediction itself $\hat{t}_i^a$ which is the predicted time of the last lightning in the alert.

Now, we will estimate two criteria : Risk and time gain.

**A gain of time** $g_i^a$ for the prediction $p^a_i$ is measured as compared with the 30 minutes baseline :

$$g_i^a = t^a + 30 - \hat{t}_i^a$$

In practice, the overall Gain $G$ of a model will be the sum of gain over all alert contained in the dataset.


**Another important aspect is the risk** (_which measures how bad it is to be wrong_) of a lightning occurring on the airport after the end of the alert. There are multiple definitions for a relevant risk. We chose to consider the number of lightnings occurring within 3 km around the airport after the end of the alert. In practice, the risk associated to a model is computed on some training data.

$$R = \frac{M^{L3}}{N^{L3}}$$

Where the lightnings of the training data are denoted as $L = \{l_i, i=1..N^L\}$ and each lightning $l_i=(d_i,t_i)$. $N^{L3}$ is the number of lightning within a 3 kilometers distance and is computed as :

$$N^{L3} = \sum_i \mathbb{1}_{d_i<3}$$

$M^{L3} $ denotes the number of missed lightning (that is to say occurring after the end of the alarm) within a 3 kilometers distance. It is computed as

$$M^{L3} = \sum_i \mathbb{1}_{d_i<3\text{ } \& \text{ }\hat{t}_i^a < t_i \text{ }\& \text{ alert}(t_i) == a } $$

with alert$(t_i)$ returns the index of the alert for the lightning $t_i$.

Intuitively, a model $m$ that satisfies $R < 0.05$ ensures that less than 5% of lightning strikes will occur outside of alert periods. 

### How are combined the two criteria, ie, risk and time gain ($R$ and $G$)

The evaluator will set an acceptable risk value : $R_{accept}=0.02$.


To compute this risk, we must somehow select a prediction $p^a_i$ from the model. For this, we define a threshold $\theta$ and we select the predicted date $\hat{t^a}$ which verifies : 

$$\hat{t}^a = min_i\text{  }  \hat{t}_i^a, s^a_i > \theta $$

Given the selected prediction $\hat{t^a}$, we can compute a gain $g^a=t^a + 30 - \hat{t}^a$. 

To set the value of the threshold $\theta$, we test different values, and, among the values which provide a $R<R_{accept}$, we select the one with the highest gain $g^a$


### Provided python implementation

The rest of the notebook is organised as follows :

- Part 1 will generate fake predictions so that we can simulate an evaluation

- Part 2 will test different threshold $\theta$ and select the one with maximum gain $G$ while respecting the contraint on the risk $R$.

- Part 3 will evaluate the model given a supplied Theta. This last part will correspond to the code we will run with your submission.

In [1]:
import pandas as pd
## here replace with your own file
input_file = '..\\data\\segment_alerts_all_airports_eval.csv'
df = pd.read_csv(input_file)

In [66]:
import pandas as pd
## here replace with your own file
input_file = '..\\data\\segment_alerts_all_airports_train.csv'
df_train = pd.read_csv(input_file)

In [2]:
init_vars = df.columns
init_vars = list(init_vars)

## Importations

### Ajout des features

In [3]:
import sys
import os
from importlib import reload

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import feature_engineering_function
reload(feature_engineering_function)
from src.feature_engineering_function import build_features

df,VAR1, TARGET1, IDS1, new_dummies1 = build_features(df)

✅ Typage & tri
✅ Variables temporelles
✅ Délais passés + futur strict
✅ Comptages rolling
✅ Comptages par type
✅ Taux d'activité
✅ Variables spatiales & azimut
✅ Variables amplitude
✅ Variables alerte
✅ Dynamique orage
✅ Silence & direction
✅ Centre de masse
✅ Cible créée | 188,175 lignes conservées


In [4]:
VAR = ['min_dist_5min',
 'time_since_last_CG20_2',
 'log_count_30min',
 'amplitude_change',
 'activity_decay',
 'time_since_last_intra_cloud2',
 'log_cg_count_10min',
 'min_dist_1min',
 'cg_20km',
 'log_std_amplitude_10min',
 'is_cloud_ground',
 'log_cg_count_20min',
 'burst_indicator',
 'hour',
 'storm_direction_change',
 'time_since_last_cloud_ground2',
 'std_lat_10min',
 'mean_dist_1min',
 'dist',
 'log_count_5min',
 'alert_duration',
 'delta_dist',
 'log_ic_count_10min',
 'storm_center_distance',
 'silence_30min',
 'azimuth_diff',
 'max_amplitude_10min',
 'std_azimuth_10min',
 'storm_spread',
 'month',
 'mean_azimuth_1min',
 'log_count_20min',
 'std_azimuth_1min',
 'azimuth',
 'storm_center_velocity',
 'log_ic_count_5min',
 'max_amplitude_1min',
 'storm_velocity',
 'mean_dist_10min',
 'storm_center_move',
 'mean_azimuth_10min',
 'mean_amplitude_1min',
 'cg_ratio',
 'rate_trend',
 'log_ic_count_20min',
 'min_dist_10min',
 'azimuth_change',
 'distance_trend',
 'mean_dist_5min',
 'mean_amplitude_10min',
 'std_lon_10min',
 'time_since_last_lightning2',
 'log_count_10min',
 'log_cg_count_5min',
 'activity_acceleration',
 'log_count_1min']

In [5]:
import joblib
artefacts = joblib.load('../models/xgb_cg10_artefacts.pkl')
model_xgb10   = artefacts['model']
xgb_vars10    = artefacts['vars_to_use']
# params      = artefacts['best_params']
imputer10     = artefacts['imputer']
# breaks      = artefacts['breaks']
# bin_stats   = artefacts['bin_stats']
# performance = artefacts['performance']
#######################################################
artefacts = joblib.load('../models/xgb_cg30_artefacts.pkl')
model_xgb30   = artefacts['model']
xgb_vars30    = artefacts['vars_to_use']
# params30      = artefacts['best_params']
imputer30     = artefacts['imputer']
#######################################################3
artefacts = joblib.load('../models/xgb_cg15_artefacts.pkl')
model_xgb15   = artefacts['model']
xgb_vars15   = artefacts['vars_to_use']
# params15     = artefacts['best_params']
imputer15     = artefacts['imputer']
#######################################################
artefacts = joblib.load('../models/xgb_cg15_3km_artefacts.pkl')
model_xgb15_3   = artefacts['model']
xgb_vars15_3   = artefacts['vars_to_use']
# params15_3     = artefacts['best_params']
imputer15_3     = artefacts['imputer']


In [6]:
df[VAR] = imputer10.fit_transform(df[VAR])
df_enc = pd.get_dummies(df[xgb_vars10])
df['y_10'] = df['time_to_next_cg20']>(10 * 60)
df['probas_10'] = model_xgb10.predict_proba(df_enc)[:, 1]

df['y_15'] = df['time_to_next_cg20']>(15 * 60)
df['probas_15'] = model_xgb15.predict_proba(df_enc)[:, 1]

df['y_30'] = df['time_to_next_cg20']>(30 * 60)
df['probas_30'] = model_xgb30.predict_proba(df_enc)[:, 1]

df_enc2 = pd.get_dummies(df[xgb_vars15_3])
df['y_15_3k'] = df['time_to_next_cg3']>(15 * 60)
df['probas_15_3km'] = 1- model_xgb15_3.predict_proba(df_enc2)[:, 1]

### Evaluation des modèles

In [7]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             brier_score_loss)
import numpy as np
def evaluate_models(df, models_config=None):
    """
    Évalue les performances de chaque modèle binaire et retourne un DataFrame synthétique.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contenant les colonnes y_* et probas_*.
    models_config : list[dict], optional
        Liste de configs. Par défaut, utilise les 4 modèles standard.
        Chaque dict : {'name': str, 'y_col': str, 'proba_col': str, 'threshold': float}

    Returns
    -------
    pd.DataFrame avec une ligne par modèle et les métriques en colonnes.
    """
    if models_config is None:
        models_config = [
            {'name': 'XGB 10min',      'y_col': 'y_10',    'proba_col': 'probas_10',     'threshold': 0.5},
            {'name': 'XGB 15min',      'y_col': 'y_15',    'proba_col': 'probas_15',     'threshold': 0.5},
            {'name': 'XGB 30min',      'y_col': 'y_30',    'proba_col': 'probas_30',     'threshold': 0.5},
            {'name': 'XGB 15min 3km',  'y_col': 'y_15_3k', 'proba_col': 'probas_15_3km', 'threshold': 0.5},
        ]

    rows = []
    for cfg in models_config:
        name      = cfg['name']
        y_col     = cfg['y_col']
        proba_col = cfg['proba_col']
        threshold = cfg.get('threshold', 0.5)

        # Filtrer les NaN sur y ET probas
        mask = df[y_col].notna() & df[proba_col].notna()
        y_true = df.loc[mask, y_col].astype(int).values
        y_prob = df.loc[mask, proba_col].values
        y_pred = (y_prob >= threshold).astype(int)

        n = len(y_true)
        prevalence = y_true.mean()

        rows.append({
            'Modèle':       name,
            'N':            n,
            'Prévalence':   f'{prevalence:.1%}',
            'Accuracy':     accuracy_score(y_true, y_pred),
            'Precision':    precision_score(y_true, y_pred, zero_division=0),
            'Recall':       recall_score(y_true, y_pred, zero_division=0),
            'F1':           f1_score(y_true, y_pred, zero_division=0),
            'ROC AUC':      roc_auc_score(y_true, y_prob),
            'PR AUC':       average_precision_score(y_true, y_prob),
            'PR':       average_precision_score(y_true, y_prob)/prevalence,
            'Brier':        brier_score_loss(y_true, y_prob),
            'Seuil':        threshold,
        })

    results = pd.DataFrame(rows)

    # Formater les colonnes numériques
    fmt_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC', 'PR AUC', 'Brier']
    for col in fmt_cols:
        results[col] = results[col].map(lambda x: round(x, 4))

    return results

In [8]:
SEUIL  = {'p15_3' : 0.591,
          'p30': 0.603,
          'p10' :0.585,
          'p15' : 0.58}

perf = evaluate_models(df, models_config=[
    {'name': 'XGB 10min',     'y_col': 'y_10',    'proba_col': 'probas_10',     'threshold': SEUIL['p10']},
    {'name': 'XGB 15min',     'y_col': 'y_15',    'proba_col': 'probas_15',     'threshold': SEUIL['p15']},
    {'name': 'XGB 30min',     'y_col': 'y_30',    'proba_col': 'probas_30',     'threshold': SEUIL['p30']},
    {'name': 'XGB 15min 3km', 'y_col': 'y_15_3k', 'proba_col': 'probas_15_3km', 'threshold': SEUIL['p15_3']},
])
print(perf.to_string(index=False))

       Modèle      N Prévalence  Accuracy  Precision  Recall     F1  ROC AUC  PR AUC       PR  Brier  Seuil
    XGB 10min 188175      21.7%    0.8569     0.7186  0.5577 0.6280   0.9138  0.7099 3.276862 0.1004  0.585
    XGB 15min 188175      17.9%    0.8755     0.7127  0.5072 0.5926   0.9211  0.6834 3.826531 0.0863  0.580
    XGB 30min 188175      13.0%    0.8875     0.6030  0.3915 0.4747   0.9050  0.5360 4.127292 0.0841  0.603
XGB 15min 3km 188175      81.2%    0.8391     0.8870  0.9189 0.9027   0.8278  0.9494 1.169152 0.1259  0.591


In [9]:
data = df[init_vars+['probas_10','probas_15','probas_30','probas_15_3km']].copy()

### Utilities for prediction

## Scénario proposé : Construction de la colonne `alerte`

### Objectif

Simuler le système d'alerte opérationnel sur les données historiques afin de créer une variable binaire
`alerte` (1 = alerte en cours, 0 = pas d'alerte) pour chaque ligne du dataframe.

### Logique implémentée

La simulation suit exactement la logique opérationnelle définie :

| Étape | Condition | Action |
|---|---|---|
| Règle absolue | Éclair CG ≤ 20 km détecté | Alerte 10 min immédiate |
| Modèle 30 min | `probas_30 > 0.5` | Surveillance, attente Δt = 5 min |
| Seuil 15 min | `probas_15 > 0.5` | Pas d'alerte, attente Δt = 5 min |
| Danger 3 km | `probas_15_3km ≤ 0.5` | Alerte 10 min immédiate |
| Modèle 10 min | `probas_10 ≤ 0.5` | Alerte 10 min |
| Modèle 10 min | `probas_10 > 0.5` + aucun éclair en 5 min | Alerte précaution 10 min |
| Pendant alerte | Nouveaux éclairs + `probas_10 ≤ 0.5` | Prolonger alerte 10 min |
| Pendant alerte | Nouveaux éclairs + `probas_10 > 0.5` | Fin d'alerte |
| Fin d'alerte | Dernier CG < 30 min | Prolonger 10 min |
| Fin d'alerte | Dernier CG ≥ 30 min | Alerte levée définitivement |

### Paramètres

- **Seuil de décision** : `0.5` sur toutes les probabilités
- **Δt initialisation** : 1 minutes (attente au premier éclair pour disposer d'un historique minimal)
- **Δt entre prédictions** : 5 minutes (évite les décisions précipitées)
- **Durée d'alerte** : 10 minutes par tranche

### Interprétation des probabilités

Les colonnes `probas_*` mesurent la **probabilité de survie** — c'est-à-dire la probabilité que le
prochain éclair n'arrive **pas** avant T minutes :

- `probas_30 > 0.5` → pas d'éclair CG ≤ 20 km prévu dans les 30 prochaines minutes
- `probas_15 > 0.5` → pas d'éclair CG ≤ 20 km prévu dans les 15 prochaines minutes  
- `probas_10 > 0.5` → pas d'éclair CG ≤ 20 km prévu dans les 10 prochaines minutes
- `probas_15_3km > 0.5` → pas d'éclair CG ≤ **3 km** prévu dans les 15 prochaines minutes

### Note importante

La simulation est réalisée **par aéroport indépendamment** et suit l'ordre chronologique strict
des lignes. Chaque instant de décision correspond à une ligne du dataframe (un éclair détecté).

In [41]:
from joblib import Parallel, delayed
from datetime import timedelta
import numpy as np

# ══════════════════════════════════════════════════════════════
# PARAMÈTRES
# ══════════════════════════════════════════════════════════════
# SEUIL  = {'p15_3' : 0.591,
#           'p30': 0.603,
#           'p10' :0.585,
#           'p15' : 0.58}
seuil = 0.6
SEUIL  = {'p15_3' : seuil,
          'p30': seuil,
          'p10' :seuil,
          'p15' : seuil}
ALERT_DURATION = timedelta(minutes=10)
DT_INIT        = timedelta(minutes=0.5)   # attente initiale (premier éclair)
DT_WAIT        = timedelta(minutes=5)   # attente entre prédictions
GAP_ORAGE      = timedelta(hours=1)     # séparation entre deux orages

orig_idx   = data.index.values          # index originaux dans df global
dates      = pd.to_datetime(data['date']).values.astype('datetime64[ns]')
icloud_v   = data['icloud'].values
dist_v     = data['dist'].values
p30_v      = data['probas_30'].values
p15_v      = data['probas_15'].values
p10_v      = data['probas_10'].values
p15_3km_v  = data['probas_15_3km'].values
airport    = data['airport'].iloc[0]
n          = len(data)


# ── Constantes converties une seule fois ──────────────────────
_ALERT_NS  = np.timedelta64(int(ALERT_DURATION.total_seconds()), 's')
_WAIT_NS   = np.timedelta64(int(DT_WAIT.total_seconds()), 's')
_INIT_NS   = np.timedelta64(int(DT_INIT.total_seconds()), 's')
_30MIN_S   = 1800.0
_GAP_S     = GAP_ORAGE.total_seconds()

In [42]:
def simulate_alert_system(df_airport):

    
    df   = df_airport.sort_values('date').reset_index(drop=False)

    orig_idx  = df['index'].values
    dates     = pd.to_datetime(df['date']).values.astype('datetime64[ns]')
    p30_v     = df['probas_30'].values
    p15_v     = df['probas_15'].values
    p10_v     = df['probas_10'].values
    p15_3km_v = df['probas_15_3km'].values
    airport   = df['airport'].iloc[0]
    n         = len(df)

    # ① Pré-calcul du masque CG20 (évite les appels répétés)
    cg20_mask = (~df['icloud'].values.astype(bool)) & (df['dist'].values <= 20)

    # ② Pré-calcul du dernier index CG20 connu à chaque position → O(1) lookup
    last_cg20_idx = np.full(n, -1, dtype=np.int64)
    last = -1
    for k in range(n):
        if cg20_mask[k]:
            last = k
        last_cg20_idx[k] = last

    alerte_arr = np.zeros(n, dtype=np.int8)
    alert_log  = []

    # ③ set_alert vectorisé avec searchsorted → O(log n)
    def set_alert_fast(start_i, end_np):
        j_end = min(int(np.searchsorted(dates, end_np, side='right')), n)
        if start_i + 1 < j_end:
            alerte_arr[start_i + 1:j_end] = 1
        return j_end

    # ④ find_cg20 vectorisé avec searchsorted → O(log n) + np.any
    def has_cg20_in_window(t_start, t_end, from_i):
        j0 = max(int(np.searchsorted(dates, t_start, side='left')), from_i)
        j1 = int(np.searchsorted(dates, t_end, side='right'))
        return j0 < j1 and np.any(cg20_mask[j0:j1])
    def has_lightning_in_window(t_start, t_end, from_i):
        j0 = max(int(np.searchsorted(dates, t_start, side='left')), from_i)
        j1 = int(np.searchsorted(dates, t_end, side='right'))
        return j0 < j1
    
    def safe_last(next_out, current_i):
        """Retourne le dernier index couvert par l'alerte, ou None si invalide."""
        idx = next_out - 1
        if idx <= current_i or idx >= n:
            return None
        return idx


    # ── Épisodes orageux ──────────────────────────────────────
    gaps = np.diff(dates) / np.timedelta64(1, 's')
    ep_bounds  = np.where(gaps > _GAP_S)[0] + 1
    ep_starts  = np.concatenate([[0], ep_bounds, [n]])

    for ep in range(len(ep_starts) - 1):
        ep_start = int(ep_starts[ep])
        ep_end   = int(ep_starts[ep + 1])# exclu

        i              = ep_start
        alert_end      = None
        alert_start_ts = None
        last_in_alert  = None
        next_check     = dates[ep_start] + _INIT_NS

        while i < ep_end:
            t_now = dates[i]

            # RÈGLE ABSOLUE : CG ≤ 20km hors alerte
            if cg20_mask[i] and alert_end is None:
                t_end_np       = t_now + _ALERT_NS
                alert_start_ts = pd.Timestamp(t_now)
                next_out       = set_alert_fast(i, t_end_np)
                alert_end      = t_end_np
                last_in_alert = safe_last(next_out, i)
                alert_log.append({'airport': airport, 'alert_start': pd.Timestamp(t_now),
                                      'alert_end': pd.Timestamp(alert_end), 'type': 'CG20'})
                next_check = alert_end
                i += 1
                continue

            # FIN D'ALERTE
            if alert_end is not None and t_now > alert_end:
                # print(i)
                if last_in_alert is not None and last_in_alert > ep_start:# il y a eu au moins 1 éclair durant l alerte
                    if last_in_alert >= n:
                        prolonger = False
                    else:
                        prolonger = p10_v[last_in_alert] <= SEUIL['p10']
                    new_end = dates[last_in_alert] + _ALERT_NS
                    # print('haut')
                else: # pas d eclairs durant l alerte
                    prolonger = False  # on met fin à l alerte
                    # print('bas')
                    # k = last_cg20_idx[i - 1] if i > ep_start else -1 # l index du dernier CG20
                    # if k >= ep_start: 
                    #     t_last_CG  = (t_now - dates[k]) / np.timedelta64(1, 's')
                    #     # print('t_last_CG',t_last_CG)
                    #     prolonger = (
                    #         k >= ep_start and
                    #         t_last_CG < _30MIN_S
                    #     )# est ce que ca fait déjà 30mn?
                    #     time_to_add = min(ALERT_DURATION.total_seconds(),_30MIN_S-t_last_CG)
                    #     # print('time_to_add',time_to_add)
                    #     new_end = alert_end +  np.timedelta64(int(time_to_add), 's')
                    # else:
                    #     prolonger = False
                # print(new_end )
                # print('prolonger',prolonger)
                if prolonger: 
                    #new_end = alert_end +  _ALERT_NS
                    start_i   = i-1
                    # print('start_i',start_i)
                    j_end  = set_alert_fast(start_i, new_end)
                    alert_log.append({'airport': airport, 'alert_start': pd.Timestamp(alert_end),
                                      'alert_end': pd.Timestamp(new_end), 'type': 'prolongation'})
                    alert_end     = new_end
                    last_in_alert = safe_last(j_end, i)
                    next_check = alert_end
                    continue
                else: # lever l alerte
                    # alert_log.append({'airport': airport, 'alert_start': pd.Timestamp(alert_start_ts),
                    #                   'alert_end': pd.Timestamp(alert_end), 'type': 'normale'})
                    alert_end = alert_start_ts = None
                    last_in_alert = None
                    next_check    = t_now
                    continue

            # ATTENTE
            if t_now < next_check:
                i += 1
                continue

            # ⑤ BOUCLE DE DÉCISION — étapes 1+2 fusionnées
            # le prochain viendra t il après 15mn?  30min?
            if p30_v[i] > SEUIL['p30'] and p15_v[i] > SEUIL['p15'] and p10_v[i] > SEUIL['p10']: # dans plus de 30 mn
                next_check = t_now + _INIT_NS #_WAIT_NS
                i += 1
                continue
            if p15_v[i] <= SEUIL['p15'] or p15_3km_v[i] <= SEUIL['p15_3'] or p10_v[i] <= SEUIL['p10']: # Dans moins de 15mn
                # Danger immédiat?
                if p15_3km_v[i] <= SEUIL['p15_3'] or p10_v[i] <= SEUIL['p10']:
                    t_type         = 'danger_3km' if p15_3km_v[i] <= SEUIL['p15_3'] else 'p10'
                    t_end_np       = t_now + _ALERT_NS
                    alert_start_ts = pd.Timestamp(t_now)
                    next_out = set_alert_fast(i, t_end_np)
                    alert_end     = t_end_np
                    last_in_alert = safe_last(next_out, i)
                    alert_log.append({'airport': airport, 'alert_start': alert_start_ts,
                                        'alert_end': pd.Timestamp(t_end_np), 'type': t_type})
                    i += 1
                    next_check = t_end_np
                    continue

            # p15 > seuil ou p10 > seuil : fenêtre 5min 
            start_i   = min(int(np.searchsorted(dates, t_now + _WAIT_NS, side='left')), ep_end)
            if not has_lightning_in_window(t_now, t_now + _WAIT_NS, i+1):
                
                t_end_np       = t_now + _WAIT_NS + _ALERT_NS# l alerte sera lancée à la fin des 5mn
                alert_start_ts = pd.Timestamp(t_now + _WAIT_NS)
                next_out = set_alert_fast(start_i, t_end_np)
                alert_end      = t_end_np
                last_in_alert = safe_last(next_out, i)
                alert_log.append({'airport': airport, 'alert_start': alert_start_ts,
                                        'alert_end': pd.Timestamp(t_end_np), 'type': 'p10'})
                next_check = alert_end
                i+=1
                continue
            else:
                next_check = t_now + _INIT_NS
                i += 1
                continue


        # if alert_end is not None:
        #     alert_log.append({'airport': airport,
        #                       'alert_start': pd.Timestamp(alert_start_ts) if alert_start_ts else None,
        #                       'alert_end': pd.Timestamp(alert_end), 'type': 'fin_episode'})

    return pd.Series(alerte_arr.astype(int), index=orig_idx), alert_log



In [43]:
print("=== Simulation du système d'alerte ===\n")
data = data.reset_index(drop=True)
groups  = [(airport, grp) for airport, grp in data.groupby('airport')]

# ⑥ Parallélisation inter-aéroports
results = Parallel(n_jobs=-1, backend='loky')(
    delayed(simulate_alert_system)(grp) for _, grp in groups
)

all_alerts = pd.Series(0, index=data.index, dtype=int)
all_logs   = []
for (airport, grp), (result, logs) in zip(groups, results):
    all_alerts.loc[result.index] = result.values
    all_logs.extend(logs)
    n_alert = result.sum()
    print(f"  {airport:10} : {len(grp):>7,} lignes → {n_alert:>6,} en alerte ({n_alert/len(grp)*100:.1f}%)")

data['alerte'] = all_alerts

df_alerts = pd.DataFrame(all_logs)
df_alerts['duree_min'] = (
    (df_alerts['alert_end'] - df_alerts['alert_start']).dt.total_seconds() / 60
).round(1)

print(f"\n=== Log : {len(df_alerts)} alertes ===")
print(df_alerts.groupby(['airport', 'type']).size().unstack(fill_value=0))
print(f"\nDurées (min) :\n{df_alerts['duree_min'].describe().round(1)}")
print(f"\nTaux d'alerte global : {data['alerte'].mean()*100:.1f}%")
print(data.groupby('airport')['alerte'].mean().mul(100).round(1).sort_values(ascending=False).to_string())


=== Simulation du système d'alerte ===

  Ajaccio    :  21,331 lignes → 17,326 en alerte (81.2%)
  Bastia     :  42,291 lignes → 37,419 en alerte (88.5%)
  Biarritz   :  50,477 lignes → 46,405 en alerte (91.9%)
  Nantes     :   7,556 lignes →  6,067 en alerte (80.3%)
  Pise       :  66,520 lignes → 61,565 en alerte (92.6%)

=== Log : 6288 alertes ===
type      CG20  danger_3km  p10  prolongation
airport                                      
Ajaccio    162           1  353           554
Bastia     260          10  479           881
Biarritz   176           6  459           666
Nantes      88           0  143           238
Pise       231          25  577           979

Durées (min) :
count    6288.0
mean        9.4
std         1.4
min         0.0
25%         9.4
50%        10.0
75%        10.0
max        10.0
Name: duree_min, dtype: float64

Taux d'alerte global : 89.7%
airport
Pise        92.6
Biarritz    91.9
Bastia      88.5
Ajaccio     81.2
Nantes      80.3


In [44]:
df_alerts_active = df_alerts.copy()
df_alerts_active = df_alerts_active.sort_values(['airport', 'alert_start']).reset_index(drop=True)

# ── 1. Fusionner les alertes qui se chevauchent / se prolongent ──
merged_alerts = []
for airport, grp in df_alerts_active.groupby('airport'):
    grp = grp.sort_values('alert_start')
    current_start = None
    current_end   = None

    for _, row in grp.iterrows():
        if current_start is None:
            current_start = row['alert_start']
            current_end   = row['alert_end']
        elif row['alert_start'] <= current_end:
            # Chevauchement ou prolongation → étendre
            current_end = max(current_end, row['alert_end'])
        else:
            merged_alerts.append({'airport': airport,
                                  'alert_start': current_start,
                                  'alert_end': current_end})
            current_start = row['alert_start']
            current_end   = row['alert_end']

    if current_start is not None:
        merged_alerts.append({'airport': airport,
                              'alert_start': current_start,
                              'alert_end': current_end})

df_merged = pd.DataFrame(merged_alerts)
print(f"Alertes brutes : {len(df_alerts_active)} → fusionnées : {len(df_merged)}")
df_merged.head()


Alertes brutes : 6288 → fusionnées : 2745


,airport,alert_start,alert_end
0,Ajaccio,2023-01-08 23:18:19,2023-01-08 23:28:19
1,Ajaccio,2023-01-08 23:35:46,2023-01-08 23:45:46
2,Ajaccio,2023-01-09 01:07:52,2023-01-09 01:17:52
3,Ajaccio,2023-01-09 01:35:18,2023-01-09 01:45:18
4,Ajaccio,2023-01-17 07:17:34,2023-01-17 07:27:34


In [45]:
# ── 2. Mapping vectorisé avec searchsorted ───────────────────
data['alert_active_start'] = pd.NaT
data['alert_active_end']   = pd.NaT
#df_merged = df_alerts.copy()
for airport, grp_alerts in df_merged.groupby('airport'):
    mask     = data['airport'] == airport
    dates_np = data.loc[mask, 'date'].values.astype('datetime64[ns]')

    starts = grp_alerts['alert_start'].values.astype('datetime64[ns]')
    ends   = grp_alerts['alert_end'].values.astype('datetime64[ns]')

    idx_insert = np.searchsorted(starts, dates_np, side='right') - 1

    valid = idx_insert >= 0
    valid[valid] &= dates_np[valid] <= ends[idx_insert[valid]]

    result_start = np.full(len(dates_np), np.datetime64('NaT'), dtype='datetime64[ns]')
    result_end   = np.full(len(dates_np), np.datetime64('NaT'), dtype='datetime64[ns]')

    result_start[valid] = starts[idx_insert[valid]]
    result_end[valid]   = ends[idx_insert[valid]]

    data.loc[mask, 'alert_active_start'] = result_start
    data.loc[mask, 'alert_active_end']   = result_end

print(f"Lignes en alerte avec dates : {data['alert_active_end'].notna().sum():,}")
print(f"Lignes hors alerte          : {data['alert_active_end'].isna().sum():,}")

Lignes en alerte avec dates : 171,255
Lignes hors alerte          : 16,920


## Scénario de base

In [46]:
def simulate_baseline_cg20(df_airport):
    """
    Baseline simple : lance une alerte de 30 min à chaque éclair CG ≤ 20 km.
    Si un nouveau CG20 survient pendant l'alerte, elle est prolongée de 30 min
    à partir de ce nouvel éclair.
    """
    df = df_airport.sort_values('date').reset_index(drop=False)

    orig_idx = df['index'].values
    dates    = pd.to_datetime(df['date']).values.astype('datetime64[ns]')
    n        = len(df)
    airport  = df['airport'].iloc[0]

    cg20_mask  = (~df['icloud'].values.astype(bool)) & (df['dist'].values <= 20)
    _30MIN_NS  = np.timedelta64(30, 'm')

    alerte_arr = np.zeros(n, dtype=np.int8)
    alert_log  = []

    alert_end      = None
    alert_start_ts = None

    for i in range(n):
        t_now = dates[i]
                # Marquer en alerte si on est dans la fenêtre
        if alert_end is not None:
            if t_now <= alert_end :#and pd.Timestamp(t_now) != alert_start_ts:
                alerte_arr[i] = 1
            else:
                # L'alerte est expirée
                alert_end = None
                alert_start_ts = None

        # CG ≤ 20 km → lancer ou prolonger
        if cg20_mask[i]:
            new_end = t_now + _30MIN_NS

            if alert_end is None:
                # Nouvelle alerte
                alert_start_ts = pd.Timestamp(t_now)
                alert_end = new_end
                alert_log.append({
                    'airport': airport,
                    'alert_start': alert_start_ts,
                    'alert_end': pd.Timestamp(alert_end),
                    'type': 'CG20'
                })
            else:
                # Prolonger si le nouveau end dépasse l'ancien
                if new_end > alert_end:
                    alert_log.append({
                        'airport': airport,
                        'alert_start': pd.Timestamp(t_now),
                        'alert_end': pd.Timestamp(new_end),
                        'type': 'prolongation'
                    })
                    alert_start_ts = pd.Timestamp(t_now)
                    alert_end = new_end



    return pd.Series(alerte_arr.astype(int), index=orig_idx), alert_log

In [47]:
print("=== Simulation du système d'alerte de base: 30mn ===\n")
data = data.reset_index(drop=True)
groups  = [(airport, grp) for airport, grp in data.groupby('airport')]

# ⑥ Parallélisation inter-aéroports
results = Parallel(n_jobs=-1, backend='loky')(
    delayed(simulate_baseline_cg20)(grp) for _, grp in groups
)

all_alerts = pd.Series(0, index=data.index, dtype=int)
all_logs   = []
for (airport, grp), (result, logs) in zip(groups, results):
    all_alerts.loc[result.index] = result.values
    all_logs.extend(logs)
    n_alert = result.sum()
    print(f"  {airport:10} : {len(grp):>7,} lignes → {n_alert:>6,} en alerte ({n_alert/len(grp)*100:.1f}%)")

data['alerte_base'] = all_alerts

df_alerts_base = pd.DataFrame(all_logs)
df_alerts_base['duree_min'] = (
    (df_alerts_base['alert_end'] - df_alerts_base['alert_start']).dt.total_seconds() / 60
).round(1)

print(f"\n=== Log : {len(df_alerts_base)} alertes ===")
print(df_alerts_base.groupby(['airport', 'type']).size().unstack(fill_value=0))
print(f"\nDurées (min) :\n{df_alerts_base['duree_min'].describe().round(1)}")
print(f"\nTaux d'alerte global : {data['alerte'].mean()*100:.1f}%")
print(data.groupby('airport')['alerte'].mean().mul(100).round(1).sort_values(ascending=False).to_string())


=== Simulation du système d'alerte de base: 30mn ===

  Ajaccio    :  21,331 lignes → 15,193 en alerte (71.2%)
  Bastia     :  42,291 lignes → 35,356 en alerte (83.6%)
  Biarritz   :  50,477 lignes → 43,927 en alerte (87.0%)
  Nantes     :   7,556 lignes →  5,732 en alerte (75.9%)
  Pise       :  66,520 lignes → 59,759 en alerte (89.8%)

=== Log : 17379 alertes ===
type      CG20  prolongation
airport                     
Ajaccio    203          2205
Bastia     263          4222
Biarritz   242          2947
Nantes      98           731
Pise       275          6193

Durées (min) :
count    17379.0
mean        30.0
std          0.0
min         30.0
25%         30.0
50%         30.0
75%         30.0
max         30.0
Name: duree_min, dtype: float64

Taux d'alerte global : 89.7%
airport
Pise        92.6
Biarritz    91.9
Bastia      88.5
Ajaccio     81.2
Nantes      80.3


In [48]:
df_alerts_active = df_alerts_base.copy()
df_alerts_active = df_alerts_active.sort_values(['airport', 'alert_start']).reset_index(drop=True)

# ── 1. Fusionner les alertes qui se chevauchent / se prolongent ──
merged_alerts = []
for airport, grp in df_alerts_active.groupby('airport'):
    grp = grp.sort_values('alert_start')
    current_start = None
    current_end   = None

    for _, row in grp.iterrows():
        if current_start is None:
            current_start = row['alert_start']
            current_end   = row['alert_end']
        elif row['alert_start'] <= current_end:
            # Chevauchement ou prolongation → étendre
            current_end = max(current_end, row['alert_end'])
        else:
            merged_alerts.append({'airport': airport,
                                  'alert_start': current_start,
                                  'alert_end': current_end})
            current_start = row['alert_start']
            current_end   = row['alert_end']

    if current_start is not None:
        merged_alerts.append({'airport': airport,
                              'alert_start': current_start,
                              'alert_end': current_end})

df_merged = pd.DataFrame(merged_alerts)
print(f"Alertes brutes : {len(df_alerts_active)} → fusionnées : {len(df_merged)}")
df_merged.head()


Alertes brutes : 17379 → fusionnées : 1081


,airport,alert_start,alert_end
0,Ajaccio,2023-01-09 01:07:52,2023-01-09 01:37:52
1,Ajaccio,2023-01-17 07:17:34,2023-01-17 07:53:30
2,Ajaccio,2023-01-17 10:52:24,2023-01-17 11:36:42
3,Ajaccio,2023-01-17 23:47:55,2023-01-18 00:40:37
4,Ajaccio,2023-01-27 23:24:37,2023-01-27 23:54:37


In [49]:
# ── 2. Mapping vectorisé avec searchsorted ───────────────────
data['alert_30_start'] = pd.NaT
data['alert_30_end']   = pd.NaT
#df_merged = df_alerts_base.copy()
for airport, grp_alerts in df_merged.groupby('airport'):
    mask     = data['airport'] == airport
    dates_np = data.loc[mask, 'date'].values.astype('datetime64[ns]')

    starts = grp_alerts['alert_start'].values.astype('datetime64[ns]')
    ends   = grp_alerts['alert_end'].values.astype('datetime64[ns]')

    idx_insert = np.searchsorted(starts, dates_np, side='right') - 1

    valid = idx_insert >= 0
    valid[valid] &= dates_np[valid] <= ends[idx_insert[valid]]

    result_start = np.full(len(dates_np), np.datetime64('NaT'), dtype='datetime64[ns]')
    result_end   = np.full(len(dates_np), np.datetime64('NaT'), dtype='datetime64[ns]')

    result_start[valid] = starts[idx_insert[valid]]
    result_end[valid]   = ends[idx_insert[valid]]

    data.loc[mask, 'alert_30_start'] = result_start
    data.loc[mask, 'alert_30_end']   = result_end

print(f"Lignes en alerte avec dates : {data['alert_30_end'].notna().sum():,}")
print(f"Lignes hors alerte          : {data['alert_30_end'].isna().sum():,}")

Lignes en alerte avec dates : 161,095
Lignes hors alerte          : 27,080


## Comparaison

In [50]:
def adjust_model_alerts(data):
    """
    Crée 3 nouvelles colonnes (alerte_adj, alert_adj_start, alert_adj_end) :
    - Exclut les alertes modèle lancées sans CG20 qui n'en couvrent aucun
    - Si l'alerte couvre un CG20 mais a été lancée avant, alert_adj_start = date du 1er CG20
    """
    data['alerte_adj']     = data['alerte'].copy()
    data['alert_adj_start'] = data['alert_active_start'].copy()
    data['alert_adj_end']   = data['alert_active_end'].copy()

    cg20 = (~data['icloud'].astype(bool)) & (data['dist'] <= 20)
    cg20_vals = cg20.values
    dates_np = data['date'].values.astype('datetime64[ns]')
    airports = data['airport'].values
    alerte_vals = data['alerte'].values

    # Alertes uniques du modèle
    unique_alerts = (
        data[data['alert_active_end'].notna()][['alert_active_start', 'alert_active_end', 'airport']]
        .drop_duplicates()
        .sort_values(['airport', 'alert_active_start'])
        .reset_index(drop=True)
    )

    for airport, grp in unique_alerts.groupby('airport'):
        ap_mask = airports == airport
        ap_idx = np.where(ap_mask)[0]
        ap_dates = dates_np[ap_idx]
        ap_cg20 = cg20_vals[ap_idx]

        starts = grp['alert_active_start'].values.astype('datetime64[ns]')
        ends   = grp['alert_active_end'].values.astype('datetime64[ns]')

        for k in range(len(starts)):
            j0 = np.searchsorted(ap_dates, starts[k], side='left')
            j1 = np.searchsorted(ap_dates, ends[k], side='right')

            w_cg20 = ap_cg20[j0:j1]
            global_idx = ap_idx[j0:j1]

            if not np.any(w_cg20):
                # Aucun CG20 dans la fenêtre → supprimer l'alerte
                data.loc[global_idx, 'alerte_adj']     = 0
                data.loc[global_idx, 'alert_adj_start'] = pd.NaT
                data.loc[global_idx, 'alert_adj_end']   = pd.NaT
            else:
                # Il y a des CG20 → recaler le start sur le 1er CG20
                first_cg20_pos = np.argmax(w_cg20)  # premier True
                first_cg20_date = ap_dates[j0 + first_cg20_pos]

                if first_cg20_date > starts[k]:
                    # Lignes avant le 1er CG20 → hors alerte
                    before_cg20 = global_idx[:first_cg20_pos]
                    data.loc[before_cg20, 'alerte_adj']     = 0
                    data.loc[before_cg20, 'alert_adj_start'] = pd.NaT
                    data.loc[before_cg20, 'alert_adj_end']   = pd.NaT

                    # Lignes à partir du CG20 → recaler le start
                    from_cg20 = global_idx[first_cg20_pos:]
                    data.loc[from_cg20, 'alert_adj_start'] = pd.Timestamp(first_cg20_date)

    n_removed = (data['alerte'] != data['alerte_adj']).sum()
    print(f"Lignes modifiées : {n_removed:,}")
    print(f"Alertes avant    : {data['alerte'].sum():,}")
    print(f"Alertes après    : {data['alerte_adj'].sum():,}")

    return data

In [51]:
data = adjust_model_alerts(data)

Lignes modifiées : 16,351
Alertes avant    : 168,782
Alertes après    : 152,431


In [52]:
def merge_overlapping_alerts(data, col_start, col_end):
    """Fusionne les alertes qui se chevauchent ou se touchent."""
    unique_alerts = (
        data[data[col_end].notna()][[col_start, col_end, 'airport']]
        .drop_duplicates()
        .sort_values(['airport', col_start])
        .reset_index(drop=True)
    )

    merged = []
    for airport, grp in unique_alerts.groupby('airport'):
        current_start = None
        current_end   = None

        for _, row in grp.iterrows():
            if current_start is None:
                current_start = pd.to_datetime(row[col_start],utc=True)
                current_end   = pd.to_datetime(row[col_end],utc=True)
            elif  pd.to_datetime(row[col_start],utc=True) <= current_end:
                current_end = max(current_end, pd.to_datetime(row[col_end],utc=True))
            else:
                merged.append({'airport': airport, col_start: current_start, col_end: current_end})
                current_start = pd.to_datetime(row[col_start],utc=True)
                current_end   = pd.to_datetime(row[col_end],utc=True)

        if current_start is not None:
            merged.append({'airport': airport, col_start: current_start, col_end: current_end})

    return pd.DataFrame(merged)

In [53]:
def compare_alert_systems(data, merged_alert=False):
    """
    Compare le système modèle (alerte) vs baseline (alerte_base).
    Version optimisée avec searchsorted (O(n log n) au lieu de O(n × alertes)).
    """
    cg20 = (~data['icloud'].astype(bool)) & (data['dist'] <= 20)
    n_cg20 = cg20.sum()
    cg20_vals = cg20.values
    dates_np = data['date'].values.astype('datetime64[ns]')
    airports = data['airport'].values

    configs = [
        {'name': 'Modèle', 'col_alerte': 'alerte_adj', 'col_start': 'alert_adj_start', 'col_end': 'alert_adj_end'},
        {'name': 'Baseline',      'col_alerte': 'alerte_base', 'col_start': 'alert_30_start',  'col_end': 'alert_30_end'},
    ]

    results = {}

    for cfg in configs:
        col_a, col_s, col_e = cfg['col_alerte'], cfg['col_start'], cfg['col_end']

        # ── Alertes uniques / fusionnées ──
        if merged_alert:
            unique_alerts = merge_overlapping_alerts(data, col_s, col_e)
        else:
            unique_alerts = (
                data[data[col_e].notna()][[col_s, col_e, 'airport']]
                .drop_duplicates()
                .sort_values(['airport', col_s])
                .reset_index(drop=True)
            )
            #unique_alerts = unique_alerts.rename(columns={col_s: 'start', col_e: 'end'})

        unique_alerts['duree_min'] = (
            (pd.to_datetime(unique_alerts[col_e],utc=True) - pd.to_datetime(unique_alerts[col_s],utc=True))
            .dt.total_seconds() / 60
        )
        n_alertes = len(unique_alerts)

        # ── POD ──
        alerte_vals = data[col_a].values
        cg20_covered = int(np.sum(cg20_vals & (alerte_vals == 1)))
        missed = n_cg20 - cg20_covered
        pod = cg20_covered / n_cg20 if n_cg20 > 0 else 0
        taux = alerte_vals.mean()

        # ── FAR vectorisé par aéroport avec searchsorted ──
        false_alerts = 0

        for airport, grp_alerts in unique_alerts.groupby('airport'):
            # Pré-filtrer une seule fois par aéroport
            ap_mask = airports == airport
            ap_idx = np.where(ap_mask)[0]
            ap_dates = dates_np[ap_idx]
            ap_cg20 = cg20_vals[ap_idx]
            ap_alerte = alerte_vals[ap_idx]

            starts = grp_alerts[col_s].values.astype('datetime64[ns]')
            ends = grp_alerts[col_e].values.astype('datetime64[ns]')

            for k in range(len(starts)):
                j0 = np.searchsorted(ap_dates, starts[k], side='left')
                j1 = np.searchsorted(ap_dates, ends[k], side='right')

                w_cg20 = ap_cg20[j0:j1]

                # Aucun CG20 dans la fenêtre globale
                # if not np.any(w_cg20):
                #     # Aucun CG20 couvert pendant l'alerte active non plus
                if not np.any(w_cg20 & (ap_alerte[j0:j1] == 1)):
                    false_alerts += 1

        far = false_alerts / n_alertes if n_alertes > 0 else 0
        csi = cg20_covered / (cg20_covered + missed + false_alerts) if (cg20_covered + missed + false_alerts) > 0 else 0
        duree_totale_h = unique_alerts['duree_min'].sum() / 60

        results[cfg['name']] = {
            'N CG20':           int(n_cg20),
            'CG20 couverts':    int(cg20_covered),
            'CG20 manqués':     int(missed),
            'POD':              round(pod, 4),
            'N alertes':        int(n_alertes),
            'Fausses alertes':  int(false_alerts),
            'FAR':              round(far, 4),
            'CSI':              round(csi, 4),
            'Taux alerte':      f'{taux:.1%}',
            'Durée moy (min)':  round(unique_alerts['duree_min'].mean(), 1),
            'Durée méd (min)':  round(unique_alerts['duree_min'].median(), 1),
            'Durée totale (h)': round(duree_totale_h, 1),
        }

    df_results = pd.DataFrame(results)

    print('=' * 55)
    print('  COMPARAISON : MODÈLE vs BASELINE')
    print('=' * 55)
    print(df_results.to_string())

    m, b = results['Modèle'], results['Baseline']
    print(f'\n{"─" * 55}')
    print(f'  Δ POD          : {m["POD"] - b["POD"]:+.4f}')
    print(f'  Δ FAR          : {b["FAR"] - m["FAR"]:+.4f}')
    print(f'  Δ CSI          : {m["CSI"] - b["CSI"]:+.4f}')
    print(f'  Δ CG20 manqués : {m["CG20 manqués"] - b["CG20 manqués"]:+d}')
    print(f'  Δ Durée tot.   : {m["Durée totale (h)"] - b["Durée totale (h)"]:+.1f} h')
    print('=' * 55)

    return df_results

In [54]:
#sans fusion
comparison = compare_alert_systems(data)

  COMPARAISON : MODÈLE vs BASELINE
                  Modèle Baseline
N CG20             18010    18010
CG20 couverts      17059    16929
CG20 manqués         951     1081
POD               0.9472     0.94
N alertes           1533     1081
Fausses alertes      484      379
FAR               0.3157   0.3506
CSI               0.9224   0.9206
Taux alerte        81.0%    85.0%
Durée moy (min)     26.7     59.8
Durée méd (min)     13.3     41.0
Durée totale (h)   681.8   1076.8

───────────────────────────────────────────────────────
  Δ POD          : +0.0072
  Δ FAR          : +0.0349
  Δ CSI          : +0.0018
  Δ CG20 manqués : -130
  Δ Durée tot.   : -395.0 h


## Part 1 Generate predictions

We simply generate 20 predictions around the true date of the last lightning with a linear growth of the confidence threshold.

In [55]:
# ── Filtrer : garder uniquement les lignes avec une alerte prédite ──
data['score'] = 1- data['probas_10']
df_submit = data[data['airport_alert_id'].notna()].copy()

# ── Ne garder que les colonnes utiles et renommer ──
df_submit = df_submit[['airport', 'airport_alert_id', 'date', 'alert_active_end', 'score']].rename(columns={
    'date':               'prediction_date',
    'alert_active_end':   'predicted_date_end_alert',
    'score':             'confidence'
})

df_submit = df_submit.reset_index(drop=True)
print(f"Shape : {df_submit.shape}")
df_submit.head()

Shape : (18010, 5)


,airport,airport_alert_id,prediction_date,predicted_date_end_alert,confidence
0,Ajaccio,531.0,2023-01-09 01:07:52+00:00,2023-01-09 01:17:52,0.332067
1,Ajaccio,532.0,2023-01-17 07:17:34+00:00,2023-01-17 07:27:34,0.164643
2,Ajaccio,532.0,2023-01-17 07:21:06+00:00,2023-01-17 07:27:34,0.665430
3,Ajaccio,532.0,2023-01-17 07:23:30+00:00,2023-01-17 07:27:34,0.426871
4,Ajaccio,533.0,2023-01-17 10:52:24+00:00,2023-01-17 11:06:10,0.715228


In [56]:
# df_alerts_active2 = df_alerts_base.copy()
# df_alerts_active2 = df_alerts_active2.sort_values(['airport', 'alert_start']).reset_index(drop=True)

# # ── 1. Fusionner les alertes qui se chevauchent / se prolongent ──
# merged_alerts = []
# for airport, grp in df_alerts_active2.groupby('airport'):
#     grp = grp.sort_values('alert_start')
#     current_start = None
#     current_end   = None

#     for _, row in grp.iterrows():
#         if current_start is None:
#             current_start = row['alert_start']
#             current_end   = row['alert_end']
#         elif row['alert_start'] <= current_end:
#             # Chevauchement ou prolongation → étendre
#             current_end = max(current_end, row['alert_end'])
#         else:
#             merged_alerts.append({'airport': airport,
#                                   'alert_start': current_start,
#                                   'alert_end': current_end})
#             current_start = row['alert_start']
#             current_end   = row['alert_end']

#     if current_start is not None:
#         merged_alerts.append({'airport': airport,
#                               'alert_start': current_start,
#                               'alert_end': current_end})

# df_merged = pd.DataFrame(merged_alerts)
# print(f"Alertes brutes : {len(df_alerts_active2)} → fusionnées : {len(df_merged)}")



### Part 2 Evaluate predictions

In [57]:
import pandas as pd
from tqdm import tqdm

In [58]:
# our 30 minutes baseline
MAX_GAP_MINUTES = 30


# here replace with your own file
# input_file = "../data/segment_alerts_all_airports_train.csv"
# df = pd.read_csv(input_file)


#building the Thetas
n_samples = 20
thetas = [i/n_samples for i in range(n_samples)]

# min distance at which we consider a lightning is really dangerous for the airport
min_dist = 3
# total number of dangerous lightning
tot_lightnings = len(df.loc[df.dist<min_dist])

# read prediction file and convert to timestampts
predictions = df_submit
predictions.predicted_date_end_alert = pd.to_datetime(predictions.predicted_date_end_alert)

# group by alerts 
alerts = data.groupby(['airport','airport_alert_id'])

The following cell computes the number of missed dangerous lightning and the gain in time for the different thetas

In [59]:
pred_over_theta = predictions#.loc[predictions['confidence'] >= theta ]
pred_over_theta_min = pred_over_theta.groupby(['airport','airport_alert_id']).predicted_date_end_alert.max()#min()


In [60]:
results = {}
min_dist = 20
#for theta in tqdm(thetas):
pred_over_theta = predictions#.loc[predictions['confidence'] >= theta ]
pred_over_theta_min = pred_over_theta.groupby(['airport','airport_alert_id']).predicted_date_end_alert.max()#min()
gain, missed_lights = 0, 0
for (airport, alert_id), end_alert_pred in pred_over_theta_min.items():
    end_pred = pd.to_datetime(end_alert_pred, utc=True)
    lightings = alerts.get_group((airport, alert_id))
    end_alert_baseline = pd.to_datetime(lightings.date, utc=True).max() + pd.Timedelta(minutes=MAX_GAP_MINUTES)
    gain += (end_alert_baseline - end_pred).total_seconds()
    missed_lights += sum(pd.to_datetime(lightings.loc[lightings.dist<min_dist].date, utc=True) > end_pred )
#results[theta] = (gain, missed_lights)
(gain, missed_lights)

(1192978.0, 0)

In [61]:
gain/3600

331.3827777777778

Perform some visualisations : normally, the higher the Theta, the lower the gain and the lower the risk. This kind of visualisation can be usefull to show to the Jury the different compromises that your model can reach.

In [62]:
# import matplotlib.pyplot as plt
# gains = [results[theta][0]/3600 for theta in thetas]
# missed = [results[theta][1] / tot_lightnings for theta in thetas]
# plt.plot(missed, gains, marker='*', markersize=8)
# for t,g,m in zip(thetas,gains,missed):
#     plt.text(m, g, str(t))
# plt.xlabel('Number of missed lightnings')
# plt.ylabel('Gain of time in hours')
# plt.title('Gain of time and missed lightnings for different thresholds')


**Selection of the best theta**

What is the theta with maximize the gain of time and respect the constraint on the acceptable risk ?

In [ ]:
# ACCEPTABLE_RISK = 0.02

# (gain_of_time, theta, missing_lightnings) = max([ (gain_of_time, theta, missing_rate)  
#      for (theta,(gain_of_time, missing_rate)) 
#      in results.items() if missing_rate/tot_lightnings < ACCEPTABLE_RISK ])

ValueError: max() iterable argument is empty

In [ ]:
# print('The system enables to save',
#       int(gain_of_time/3600),
#       'hours, with a risk of',ACCEPTABLE_RISK,
#       'by using a threshold on the confidence score with a value of',
#       theta)

### Part 3 Evaluate given a Theta

It is essentially the same code as before, its just that we evaluate it with one value of Theta. So you don't really need this code yourself. But this is what we will do with the predictions.csv file you might send us.


In [64]:
# redefining some variables
import pandas as pd
#predictions = pd.read_csv('predictions.csv')
predictions.predicted_date_end_alert = pd.to_datetime(predictions.predicted_date_end_alert)
MAX_GAP_MINUTES = 30
# input_file = "/home/paul/programmation/iapau/databattle2026/databattle2026/data_train_databattle2026_v1/segment_alerts_all_airports_train.csv"
# df = pd.read_csv(input_file)
min_dist = 3 # at which distance a strike is considered dangerous
tot_lightnings = len(df.loc[df.dist<min_dist])
alerts = df.groupby(['airport','airport_alert_id'])

theta = 0.4 # using the theta value we obtained from the previous section

In [65]:
# Essentially the same code as in part 2 of this notebook

pred_over_theta = predictions.loc[predictions['confidence'] >= theta ]
pred_over_theta_min = pred_over_theta.groupby(['airport','airport_alert_id']).predicted_date_end_alert.max()
gain, missed_lights = 0, 0
for (airport, alert_id), end_alert_pred in pred_over_theta_min.items():
    lightings = alerts.get_group((airport, alert_id))
    end_alert_baseline = pd.to_datetime(lightings.date, utc=True).max() + pd.Timedelta(minutes=MAX_GAP_MINUTES)
    gain += (pd.to_datetime(end_alert_baseline,utc=True) - pd.to_datetime(end_alert_pred,utc=True)).total_seconds()
    missed_lights += sum(pd.to_datetime(lightings.loc[lightings.dist<min_dist].date, utc=True) > pd.to_datetime(end_alert_pred ,utc=True))

print('for a theta of',theta,
      'the gain of time is',int(gain/3600),
      'hours, with a rate of missing lightnings',missed_lights/tot_lightnings)

for a theta of 0.4 the gain of time is 270 hours, with a rate of missing lightnings 0.0


### Conclusion

Feel free to play and modify this notebook in any way you like or judge relevant. 

The distance of 3km was suggested by Sébastien from Meteorage, but it might be interesting to provide results for other values as well.

One thing we wondered but did not consider in the notebook is to compute a gain, not only from the 30 minutes baseline, but also by taking into account the moment of the prediction. Intuitively, for the same prediction, it is better to send the information as soon as possible. With our definition, the Gain is measured only between the baseline and the value of the prediction, and the date is not taken into account. If you worked on this aspect, that's perfectly legitimate!

